# LeetCode 1846 — Maximum Element After Decreasing and Rearranging bằng Python

Link đề: https://leetcode.com/problems/maximum-element-after-decreasing-and-rearranging/ · Độ khó: **Medium** · Chủ đề: Array, Greedy, Sorting

Skeleton LeetCode:

```python
class Solution:
    def maximumElementAfterDecrementingAndRearranging(self, arr: List[int]) -> int:
        ...
```

## 1. Mục tiêu học

1. Hiểu hai thao tác được phép (**giảm** một phần tử, **sắp lại** tùy ý) và hai điều kiện (phần tử đầu bằng 1, hai phần tử kề chênh ≤ 1).
2. Thấy rằng sau khi sắp **tăng dần**, tham lam "mỗi phần tử lớn nhất có thể" `b[i] = min(a[i], b[i−1] + 1)` là tối ưu.
3. Chứng minh bằng **hai cận trên**: đáp án `≤ n` và `≤` giá trị tham lam, cùng lập luận "hoán đổi" cho việc sắp tăng.
4. Tối ưu thành **đếm** `O(n)` (giá trị vượt `n` không có ích) và giải thêm bằng **nhị phân trên đáp án**.
5. Kiểm chứng bằng vét cạn mọi hoán vị trên mảng nhỏ.

## 2. Đề bài & ví dụ

Cho mảng số nguyên dương `arr`. Thực hiện một số thao tác (có thể không thao tác nào) để mảng thỏa:
- phần tử **đầu tiên** bằng `1`;
- `|arr[i] − arr[i−1]| ≤ 1` với mọi `1 ≤ i < n`.

Hai loại thao tác, dùng bao nhiêu lần cũng được:
- **giảm** giá trị một phần tử xuống một số nguyên dương nhỏ hơn;
- **sắp lại** các phần tử theo thứ tự bất kỳ.

Trả về **giá trị lớn nhất có thể** của một phần tử sau khi thỏa điều kiện.

**Ví dụ 1:** `arr = [2,2,1,2,1]` → `2` (sắp thành `[1,2,2,2,1]`).

**Ví dụ 2:** `arr = [100,1,1000]` → `3` (`[1,100,1000]` → giảm thành `[1,2,3]`).

**Ví dụ 3:** `arr = [1,2,3,4,5]` → `5` (đã thỏa).

**Ràng buộc:** `1 ≤ n ≤ 10⁵`; `1 ≤ arr[i] ≤ 10⁹`.

In [1]:
import random
import time
from itertools import permutations
from typing import List

examples = [([2, 2, 1, 2, 1], 2), ([100, 1, 1000], 3), ([1, 2, 3, 4, 5], 5)]
for arr, exp in examples:
    print(f"arr={arr} -> kỳ vọng {exp}")

arr=[2, 2, 1, 2, 1] -> kỳ vọng 2
arr=[100, 1, 1000] -> kỳ vọng 3
arr=[1, 2, 3, 4, 5] -> kỳ vọng 5


## 3. Kiến thức nền

### 3.1 Kiểm tra một mảng hợp lệ

In [2]:
def valid(b):
    return b[0] == 1 and all(abs(b[i] - b[i - 1]) <= 1 for i in range(1, len(b)))

print(valid([1, 2, 2, 2, 1]), valid([1, 3]), valid([2, 1]), valid([1, 2, 3]))

True False False True


### 3.2 Sắp xếp và `min`

`sorted(arr)` cho thứ tự tăng; `min(x, y)` để "giảm tới mức cho phép" — vì chỉ được **giảm**, giá trị mới của phần tử `x` là `min(x, giới_hạn)`.

### 3.3 Đếm với giá trị bị chặn (`min(v, n)`)

Một mảng hợp lệ dài `n` có giá trị lớn nhất `≤ n` (bắt đầu từ 1, mỗi bước tăng tối đa 1). Vì vậy mọi `arr[i] > n` có thể coi như bằng `n` mà không mất gì — cho phép dùng **mảng đếm** kích thước `n + 1` thay vì sắp các số tới `10⁹`.

In [3]:
arr = [100, 1, 1000]; n = len(arr)
cnt = [0] * (n + 1)
for v in arr:
    cnt[min(v, n)] += 1
print("cnt =", cnt)

cnt = [0, 1, 0, 2]


### 3.4 `itertools.permutations`

Liệt kê mọi thứ tự sắp xếp (`n!`) — dùng cho vét cạn với `n ≤ 7`.

### 3.5 Tập giá trị của một mảng hợp lệ là **mọi số từ 1 tới max**

Dãy bắt đầu ở 1 và mỗi bước thay đổi tối đa 1 ⇒ để lên tới giá trị `M` phải "đi qua" mọi số `1, 2, …, M`. Vì vậy một mảng hợp lệ có giá trị lớn nhất `M` cần **ít nhất `M` phần tử**, và phần tử dùng cho bậc `k` phải có giá trị gốc `≥ k`. Đây là nguồn gốc của cận `đáp án ≤ n` và của điều kiện trong Cách 4.

In [4]:
def values_cover(b):
    return set(b) == set(range(1, max(b) + 1))

rng = random.Random(0)
cnt_checked = 0
for _ in range(2000):
    b = [1]
    for _ in range(rng.randint(0, 8)):
        b.append(max(1, b[-1] + rng.choice([-1, 0, 1])))
    assert valid(b) and values_cover(b); cnt_checked += 1
print("Mọi mảng hợp lệ ngẫu nhiên đều phủ đủ 1..max:", cnt_checked, "mảng")

Mọi mảng hợp lệ ngẫu nhiên đều phủ đủ 1..max: 2000 mảng


## 4. Ý tưởng nền tảng / trực giác

Vì được sắp lại tùy ý và chỉ được **giảm**, ta muốn đặt các phần tử thành một "cầu thang" `1, 2, 3, …` càng cao càng tốt. Phần tử nhỏ nên đặt **trước** (chúng chỉ "đủ cao" cho bậc thấp), phần tử lớn đặt **sau** (dùng cho bậc cao).

Sau khi sắp tăng, đi từ trái: bậc đầu là 1; mỗi phần tử tiếp theo cao nhất có thể là `min(a[i], bậc trước + 1)`.

Tính tay **Ví dụ 2**: sắp `[1, 100, 1000]`:

| i | a[i] | bậc trước + 1 | b[i] = min |
|---|---|---|---|
| 0 | 1 | — (phải là 1) | 1 |
| 1 | 100 | 2 | 2 |
| 2 | 1000 | 3 | 3 |

Đáp án `3`. **Ví dụ 1**: sắp `[1,1,2,2,2]` ⇒ `b = [1,1,2,2,2]` ⇒ `2`.

In [5]:
for arr, exp in examples:
    a = sorted(arr); b = [1]
    for x in a[1:]:
        b.append(min(x, b[-1] + 1))
    print(f"sắp {a} -> cầu thang {b} -> max {max(b)} (kỳ vọng {exp})")

sắp [1, 1, 2, 2, 2] -> cầu thang [1, 1, 2, 2, 2] -> max 2 (kỳ vọng 2)
sắp [1, 100, 1000] -> cầu thang [1, 2, 3] -> max 3 (kỳ vọng 3)
sắp [1, 2, 3, 4, 5] -> cầu thang [1, 2, 3, 4, 5] -> max 5 (kỳ vọng 5)


### 4.1 Thử nghiệm: thứ tự tăng dần luôn thắng mọi thứ tự khác

In [6]:
def greedy_on_order(p):
    b = 1; mx = 1
    for x in p[1:]:
        b = min(x, b + 1); mx = max(mx, b)
    return mx

rng = random.Random(1)
for _ in range(300):
    arr = [rng.randint(1, 6) for _ in range(rng.randint(1, 6))]
    best_sorted = greedy_on_order(sorted(arr))
    assert all(greedy_on_order(p) <= best_sorted for p in permutations(arr))
print("Trên 300 mảng: không hoán vị nào tốt hơn thứ tự tăng dần")
print("ví dụ [5, 1, 1]:", {p: greedy_on_order(p) for p in set(permutations([5, 1, 1]))})

Trên 300 mảng: không hoán vị nào tốt hơn thứ tự tăng dần
ví dụ [5, 1, 1]: {(5, 1, 1): 1, (1, 1, 5): 2, (1, 5, 1): 2}


## 5. Cách 1 — Brute Force / baseline: mọi hoán vị + tham lam trên từng hoán vị

**Ý tưởng:** với một thứ tự cố định, giá trị lớn nhất mỗi vị trí có thể đạt là `b[0] = 1`, `b[i] = min(p[i], b[i−1] + 1)` (giảm tối thiểu). Lấy `max(b)` cho mỗi hoán vị, rồi max trên mọi hoán vị. (Với thứ tự cố định, cách giảm này cho giá trị **lớn nhất từng vị trí** — xem mục 10 — nên đúng là tối ưu cho thứ tự đó.)

In [7]:
def max_elem_bruteforce(arr):
    best = 0
    for p in set(permutations(arr)):
        b = 1; mx = 1
        for x in p[1:]:
            b = min(x, b + 1); mx = max(mx, b)
        best = max(best, mx)
    return best

for arr, exp in examples:
    got = max_elem_bruteforce(arr); print(got, "kỳ vọng", exp); assert got == exp

2 kỳ vọng 2
3 kỳ vọng 3
5 kỳ vọng 5


**Độ phức tạp:** `O(n! · n)` — chỉ dùng với `n ≤ 8`.

### Cách 1' — Vét cạn "thật" mọi mảng đích (kiểm chứng cả luật giảm)

Cho mảng rất nhỏ, thử mọi mảng `b` với `1 ≤ b[i] ≤ arr[perm[i]]` và kiểm tra `valid(b)` — bám sát định nghĩa nhất.

In [8]:
from itertools import product
def max_elem_true_brute(arr):
    best = 0
    for p in set(permutations(arr)):
        for b in product(*[range(1, x + 1) for x in p]):
            if valid(b):
                best = max(best, max(b))
    return best

print([max_elem_true_brute(a) for a in ([2, 2, 1, 2, 1], [3, 1, 2], [5, 5])])

[2, 3, 2]


**Nút thắt:** thử mọi thứ tự trong khi thứ tự **tăng dần** luôn tốt nhất.

## 6. Ý tưởng thuật toán / bước đột phá

**Insight 1 (sắp tăng là tối ưu):** nếu trong một thứ tự có `p[i] > p[i+1]`, đổi chỗ hai phần tử không làm cầu thang tham lam tệ đi (phần tử lớn đặt sau "có nhiều chỗ để cao hơn"). Lặp lại ⇒ thứ tự tăng dần.

**Insight 2 (tham lam từng bậc):** `b[0] = 1`, `b[i] = min(a[i], b[i−1] + 1)` cho mỗi vị trí giá trị **lớn nhất có thể** ⇒ `b[n−1] = max(b)` là đáp án.

**Insight 3 (chặn bởi `n`):** đáp án `≤ n`. Giá trị `> n` được coi như `n` ⇒ **đếm** `cnt[min(v, n)]`, rồi duyệt `v = 1..n`: `ans = min(v, ans + cnt[v])` (thêm tối đa `cnt[v]` bậc nhưng không vượt `v`).

**Quy tắc:** "nếu phần tử tiếp theo cao hơn bậc hiện tại thì cầu thang lên 1; nếu không thì giữ nguyên (bằng chính nó)".

**Invariant:** sau khi xử lý `a[0..i]` (đã sắp), `b[i]` là giá trị lớn nhất mà phần tử thứ `i` có thể có trong mọi mảng hợp lệ dùng đúng các phần tử đó.

## 7. Các cách tiếp theo

### 7.1 Cách 2 — Sắp tăng + tham lam

In [9]:
def max_elem_sort(arr):
    a = sorted(arr); b = 1
    for x in a[1:]:
        b = min(x, b + 1)
    return b

print([max_elem_sort(a) for a, _ in examples])

[2, 3, 5]


**Code nộp LeetCode (Cách 2):** xem mục 9. **Độ phức tạp:** `O(n log n)`, Space `O(n)` (hoặc `O(1)` nếu sắp tại chỗ).

### 7.2 Cách 3 — Đếm (counting) `O(n)`

**Giải thích từng đoạn:**
1. `cnt[min(v, n)] += 1` cho mọi `v`.
2. `ans = 0`; duyệt `v = 1..n`: các phần tử có giá trị `v` có thể nâng cầu thang thêm tối đa `cnt[v]` bậc, nhưng cầu thang không thể vượt `v` ⇒ `ans = min(v, ans + cnt[v])`.

In [10]:
class SolutionCounting:
    def maximumElementAfterDecrementingAndRearranging(self, arr: List[int]) -> int:
        n = len(arr)
        cnt = [0] * (n + 1)
        for v in arr:
            cnt[min(v, n)] += 1
        ans = 0
        for v in range(1, n + 1):
            ans = min(v, ans + cnt[v])
        return ans

print([SolutionCounting().maximumElementAfterDecrementingAndRearranging(a) for a, _ in examples])

[2, 3, 5]


**Vì sao `ans = min(v, ans + cnt[v])`:** trước khi xét giá trị `v`, cầu thang dùng các phần tử `< v` đạt đỉnh `ans` (`≤ v − 1`). Mỗi phần tử giá trị `v` có thể đặt tiếp lên bậc `ans + 1, ans + 2, …` nhưng không được vượt `v`. Đây là phiên bản "theo nhóm" của Cách 2.

**Độ phức tạp Cách 3:** Time `O(n)`, Space `O(n)`.

### 7.3 Cách 4 — Nhị phân trên đáp án

`ok(M)`: có thể đạt phần tử lớn nhất `M` không? Cần `M` phần tử để làm các bậc `1..M`; tốt nhất dùng `M` phần tử **lớn nhất**, bậc `k` dùng phần tử nhỏ thứ `k` trong số đó: cần `a[n−M+k−1] ≥ k` với mọi `k`. Các phần tử còn lại giảm về 1 và đặt ở đầu (chuỗi `1,1,…,1,2,3,…,M` hợp lệ). `ok` đơn điệu (đạt `M` thì đạt `M−1`) ⇒ nhị phân.

In [11]:
def max_elem_binary(arr):
    a = sorted(arr); n = len(a)
    def ok(M):
        return all(a[n - M + k - 1] >= k for k in range(1, M + 1))
    lo, hi = 1, n
    while lo < hi:
        mid = (lo + hi + 1) // 2
        if ok(mid): lo = mid
        else: hi = mid - 1
    return lo

print([max_elem_binary(a) for a, _ in examples])

[2, 3, 5]


**Độ phức tạp Cách 4:** `O(n log n)` (sắp + `log n` lần kiểm tra `O(n)`). Chậm hơn Cách 2 nhưng cho một góc nhìn khác (điều kiện đủ/cần rõ ràng).

### 7.4 Cách 5 — Công thức đóng: `min(n, min_i (a[i] + (n − 1 − i)))`

Mở vòng lặp của Cách 2: `b[n−1] = min(a[n−1], b[n−2] + 1) = min(a[n−1], a[n−2] + 1, b[n−3] + 2) = …`. Thay `b[0] = 1` (tức coi `a[0]` như 1), ta được

`đáp án = min( 1 + (n − 1), min_{i ≥ 1} (a[i] + (n − 1 − i)) )`.

Ý nghĩa: phần tử `a[i]` (đã sắp) "giới hạn" đỉnh cầu thang ở mức `a[i] + (số bậc còn lại sau nó)`. Công thức này cho thấy ngay vì sao đáp án `≤ n` (số hạng đầu) và vì sao một phần tử nhỏ ở cuối kéo đáp án xuống mạnh.

In [12]:
def max_elem_formula(arr):
    a = sorted(arr); n = len(a)
    return min([n] + [a[i] + (n - 1 - i) for i in range(1, n)])

for arr, exp in examples:
    a = sorted(arr); n = len(a)
    terms = [n] + [a[i] + (n - 1 - i) for i in range(1, n)]
    print(f"{arr}: các giới hạn {terms} -> {min(terms)} (kỳ vọng {exp})")

[2, 2, 1, 2, 1]: các giới hạn [5, 4, 4, 3, 2] -> 2 (kỳ vọng 2)
[100, 1, 1000]: các giới hạn [3, 101, 1000] -> 3 (kỳ vọng 3)
[1, 2, 3, 4, 5]: các giới hạn [5, 5, 5, 5, 5] -> 5 (kỳ vọng 5)


**Độ phức tạp Cách 5:** `O(n log n)` (sắp). Kết hợp với ý tưởng đếm cũng làm được `O(n)`.

## 8. TRACE từng bước

Trace Cách 2 và Cách 3 trên ví dụ tự dựng `[3, 1, 7, 2, 2, 9]` (ví dụ tự dựng — đối chiếu brute force; nên đối chiếu lại đề gốc):

| i | a[i] (đã sắp) | b[i−1] + 1 | b[i] |
|---|---|---|---|
| 0 | 1 | — | 1 |
| 1 | 2 | 2 | 2 |
| 2 | 2 | 3 | 2 |
| 3 | 3 | 3 | 3 |
| 4 | 7 | 4 | 4 |
| 5 | 9 | 5 | **5** |

In [13]:
def trace_sort(arr):
    a = sorted(arr); b = 1
    print(f"sắp: {a}\n  i=0: b=1 (bắt buộc)")
    for i, x in enumerate(a[1:], 1):
        nb = min(x, b + 1)
        print(f"  i={i}: a[i]={x}, b_trước+1={b + 1} -> b={nb}{' (bị giới hạn bởi a[i])' if x < b + 1 else ''}")
        b = nb
    return b

def trace_count(arr):
    n = len(arr); cnt = [0] * (n + 1)
    for v in arr: cnt[min(v, n)] += 1
    ans = 0; print("cnt (đã chặn bởi n):", {v: cnt[v] for v in range(1, n + 1) if cnt[v]})
    for v in range(1, n + 1):
        new = min(v, ans + cnt[v])
        if cnt[v] or new != ans:
            print(f"  v={v}: ans = min({v}, {ans} + {cnt[v]}) = {new}")
        ans = new
    return ans

arr = [3, 1, 7, 2, 2, 9]
r1 = trace_sort(arr); print(); r2 = trace_count(arr)
assert r1 == r2 == max_elem_bruteforce(arr) == 5

sắp: [1, 2, 2, 3, 7, 9]
  i=0: b=1 (bắt buộc)
  i=1: a[i]=2, b_trước+1=2 -> b=2
  i=2: a[i]=2, b_trước+1=3 -> b=2 (bị giới hạn bởi a[i])
  i=3: a[i]=3, b_trước+1=3 -> b=3
  i=4: a[i]=7, b_trước+1=4 -> b=4
  i=5: a[i]=9, b_trước+1=5 -> b=5

cnt (đã chặn bởi n): {1: 1, 2: 2, 3: 1, 6: 2}
  v=1: ans = min(1, 0 + 1) = 1
  v=2: ans = min(2, 1 + 2) = 2
  v=3: ans = min(3, 2 + 1) = 3
  v=6: ans = min(6, 3 + 2) = 5


## 9. Code nộp LeetCode

Bản chọn nộp: **Cách 2** — sắp tăng + tham lam.

In [14]:
class Solution:
    def maximumElementAfterDecrementingAndRearranging(self, arr: List[int]) -> int:
        arr = sorted(arr)
        b = 1
        for x in arr[1:]:
            b = min(x, b + 1)
        return b

for arr, exp in examples:
    got = Solution().maximumElementAfterDecrementingAndRearranging(arr); print(got, exp); assert got == exp

2 2
3 3
5 5


## 10. Vì sao thuật toán đúng

**Bổ đề 1 (cận trên với thứ tự cố định).** Với thứ tự `p` cố định, mọi mảng hợp lệ `c` (với `1 ≤ c[i] ≤ p[i]`) thỏa `c[i] ≤ g[i]`, trong đó `g[0] = 1`, `g[i] = min(p[i], g[i−1] + 1)`. *Quy nạp:* `c[0] = 1 = g[0]`; `c[i] ≤ p[i]` và `c[i] ≤ c[i−1] + 1 ≤ g[i−1] + 1` ⇒ `c[i] ≤ g[i]`.

**Bổ đề 2 (sắp tăng tối ưu).** Gọi `a` là `arr` sắp tăng, `g` là dãy tham lam trên `a`. Với bất kỳ thứ tự `p` và mảng hợp lệ `c`, sắp các giá trị `c` tăng dần thành `c'`: `c'` vẫn hợp lệ (dãy tăng bắt đầu 1, bước nhảy ≤ 1 vì tập giá trị của một dãy hợp lệ là các số liên tiếp `1..max`), và `c'[k] ≤ a[k]` (phần tử nhỏ thứ `k` của `c` không vượt phần tử nhỏ thứ `k` của `arr`, vì `c ≤ p` từng vị trí). Áp dụng Bổ đề 1 cho thứ tự `a` ⇒ `max(c) = c'[n−1] ≤ g[n−1]`.

**Đạt được:** `g` (trên `a` sắp tăng) không giảm, bắt đầu 1, bước ≤ 1 ⇒ hợp lệ, `max = g[n−1]`. ∎

## 11. Phân tích Time / Space Complexity

Gọi `n = len(arr) ≤ 10⁵`.

| Cách | Time | Space |
|---|---|---|
| 1. Mọi hoán vị | `O(n! · n)` | `O(n)` |
| 1'. Mọi mảng đích | `O(n! · Π arr[i])` | `O(n)` |
| 2. Sắp + tham lam | `O(n log n)` | `O(n)` |
| 3. Đếm | `O(n)` | `O(n)` |
| 4. Nhị phân trên đáp án | `O(n log n)` | `O(n)` |
| 5. Công thức đóng | `O(n log n)` | `O(n)` |

In [15]:
rng = random.Random(1846)
big = [rng.randint(1, 10 ** 9) for _ in range(10 ** 5)]
small_vals = [rng.randint(1, 50) for _ in range(10 ** 5)]
for name, f in [("Cách 2", Solution().maximumElementAfterDecrementingAndRearranging), ("Cách 3", SolutionCounting().maximumElementAfterDecrementingAndRearranging), ("Cách 4", max_elem_binary)]:
    for label, data in [("giá trị lớn", big), ("giá trị ≤ 50", small_vals)]:
        t = time.perf_counter(); r = f(data); print(f"{name} {label:<12}: {r:>6} ({(time.perf_counter()-t)*1000:.0f} ms)")

Cách 2 giá trị lớn : 100000 (20 ms)
Cách 2 giá trị ≤ 50:     50 (10 ms)
Cách 3 giá trị lớn : 100000 (7 ms)
Cách 3 giá trị ≤ 50:     50 (7 ms)


Cách 4 giá trị lớn : 100000 (106 ms)
Cách 4 giá trị ≤ 50:     50 (7 ms)


## 12. Bộ test

In [16]:
ALL = {"formula": max_elem_formula, "sort": max_elem_sort, "counting": SolutionCounting().maximumElementAfterDecrementingAndRearranging,
       "binary": max_elem_binary, "submit": Solution().maximumElementAfterDecrementingAndRearranging}
def run_all(arr, brute=True):
    res = {k: f(list(arr)) for k, f in ALL.items()}
    if brute:
        res["brute"] = max_elem_bruteforce(arr)
    assert len(set(res.values())) == 1, (arr, res)
    return res["submit"]

for arr, exp in examples:
    assert run_all(arr) == exp
edge = [([1], 1), ([10 ** 9], 1), ([1, 1, 1], 1), ([5, 5, 5, 5], 4), ([2, 3, 4], 3), ([1, 3, 3, 3, 10], 4), ([10 ** 9] * 6, 6)]
for arr, exp in edge:
    assert run_all(arr) == exp, arr
rng = random.Random(18460)
for _ in range(400):
    run_all([rng.randint(1, rng.choice([3, 6, 10 ** 9])) for _ in range(rng.randint(1, 7))])
for _ in range(80):
    tiny = [rng.randint(1, 3) for _ in range(rng.randint(1, 4))]
    assert max_elem_true_brute(tiny) == Solution().maximumElementAfterDecrementingAndRearranging(tiny), tiny
for _ in range(30):
    run_all([rng.randint(1, rng.choice([100, 10 ** 9])) for _ in range(rng.randint(1, 5000))], brute=False)
print("Ví dụ, edge case, 400 test có vét cạn hoán vị, 80 test vét cạn thật, 30 test lớn: OK")

Ví dụ, edge case, 400 test có vét cạn hoán vị, 80 test vét cạn thật, 30 test lớn: OK


### 12.1 Kiểm tra tính chất

1. `1 ≤ đáp án ≤ n`.
2. **Đơn điệu:** tăng một phần tử bất kỳ không làm đáp án giảm.
3. Thêm một phần tử mới vào mảng làm đáp án tăng **tối đa 1**.

In [17]:
rng = random.Random(12)
for _ in range(500):
    arr = [rng.randint(1, 20) for _ in range(rng.randint(1, 15))]
    a0 = f_sol = Solution().maximumElementAfterDecrementingAndRearranging(arr)
    assert 1 <= a0 <= len(arr)
    j = rng.randrange(len(arr)); bigger = arr[:]; bigger[j] += rng.randint(1, 10)
    assert Solution().maximumElementAfterDecrementingAndRearranging(bigger) >= a0
    plus = arr + [rng.randint(1, 30)]
    assert a0 <= Solution().maximumElementAfterDecrementingAndRearranging(plus) <= a0 + 1
print("3 tính chất đúng trên 500 mảng ngẫu nhiên")

3 tính chất đúng trên 500 mảng ngẫu nhiên


## 13. Các lỗi người mới hay gặp

### Lỗi 1 — Không sắp xếp

In [18]:
def wrong_no_sort(arr):
    b = 1
    for x in arr[1:]:
        b = min(x, b + 1)
    return b

arr, exp = examples[0]
print("sai :", wrong_no_sort(arr), "| đúng:", exp)

sai : 1 | đúng: 2


**Vì sao sai:** thứ tự gốc `[2,2,1,2,1]` kết thúc bằng `1` ⇒ `b` cuối = 1. Vả lại "lấy `b` cuối" chỉ là max khi dãy không giảm. **Sửa:** sắp tăng trước.

### Lỗi 2 — Quên ép phần tử đầu bằng 1 (bắt đầu từ `a[0]`)

In [19]:
def wrong_first(arr):
    a = sorted(arr); b = a[0]                              # SAI: phải là 1
    for x in a[1:]:
        b = min(x, b + 1)
    return b

arr, exp = examples[1]
print("sai :", wrong_first([5, 5]), "| đúng:", Solution().maximumElementAfterDecrementingAndRearranging([5, 5]))

sai : 5 | đúng: 2


**Vì sao sai:** đề yêu cầu `arr[0] = 1`; `[5, 5]` phải thành `[1, 2]` ⇒ 2, code sai cho 5. **Sửa:** `b = 1`.

### Lỗi 3 — Trả `max(arr)` hoặc `n` một cách máy móc

In [20]:
def wrong_return_n(arr):
    return len(arr)

for arr, exp in examples:
    print("sai :", wrong_return_n(arr), "| đúng:", exp)

sai : 5 | đúng: 2
sai : 3 | đúng: 3
sai : 5 | đúng: 5


**Vì sao sai:** `n` chỉ là **cận trên**; phần tử nhỏ (như nhiều số 1, 2) làm cầu thang bị "đè" thấp. **Sửa:** tham lam thật sự.

### Lỗi 4 — Đếm nhưng không chặn giá trị bởi `n` (chỉ số vượt mảng)

In [21]:
def wrong_count_no_cap(arr):
    n = len(arr); cnt = [0] * (n + 1)
    try:
        for v in arr:
            cnt[v] += 1                                      # SAI: v có thể tới 10^9
    except IndexError as e:
        return f"IndexError: {e}"
    ans = 0
    for v in range(1, n + 1):
        ans = min(v, ans + cnt[v])
    return ans

arr, exp = examples[1]
print("sai :", wrong_count_no_cap(arr), "| đúng:", exp)

sai : IndexError: list index out of range | đúng: 3


**Vì sao sai:** `cnt[1000]` vượt mảng độ dài `n + 1 = 4`. Tạo mảng `10⁹` phần tử thì hết bộ nhớ. **Sửa:** `cnt[min(v, n)]` — giá trị trên `n` vô dụng như nhau.

### Lỗi 5 — Dùng `b + 1` mà không `min` với `a[i]` (quên rằng chỉ được giảm)

In [22]:
def wrong_no_min(arr):
    a = sorted(arr); b = 1
    for _ in a[1:]:
        b += 1                                              # SAI: phần tử không thể tăng
    return b

arr, exp = examples[0]
print("sai :", wrong_no_min(arr), "| đúng:", exp)

sai : 5 | đúng: 2


**Vì sao sai:** chỉ được **giảm**, không được tăng: phần tử 1 không thể thành 2. **Sửa:** `b = min(x, b + 1)`.

### Lỗi 6 — Trả `max(b)` đúng nhưng tính `b` trên mảng đã bị **sắp giảm dần**

In [23]:
def wrong_desc(arr):
    a = sorted(arr, reverse=True); b = 1; mx = 1          # SAI: phần tử lớn bị "lãng phí" ở bậc thấp
    for x in a[1:]:
        b = min(x, b + 1); mx = max(mx, b)
    return mx

arr = [1, 2, 3, 4, 5]
print("sai :", wrong_desc(arr), "| đúng:", Solution().maximumElementAfterDecrementingAndRearranging(arr))

sai : 3 | đúng: 5


**Vì sao sai:** sắp giảm thì phần tử nhỏ nằm cuối, "chặn" cầu thang (`[5,4,3,2,1]` ⇒ `1,2,3,2,1`, đỉnh chỉ 3). Phần tử lớn phải dành cho bậc **cao** ⇒ sắp **tăng**.

## 14. Mẹo làm bài nhanh & nhận dạng pattern

- "Được sắp lại tùy ý + chỉ được giảm + ràng buộc kề nhau" ⇒ **sắp tăng + tham lam từng bước**.
- Ràng buộc bước nhảy `≤ 1` bắt đầu từ 1 ⇒ đáp án `≤ n` ⇒ giá trị lớn có thể **chặn** về `n` ⇒ counting sort `O(n)`.
- Chứng minh tham lam bằng cận trên quy nạp (`c[i] ≤ g[i]`).
- Kiểm tra nhanh: mảng toàn số rất lớn ⇒ đáp án `n`; có số 1 lặp nhiều ⇒ đáp án nhỏ.

## 15. Biến thể của bài

1. **Không được sắp lại** (giữ thứ tự): chỉ chạy tham lam trên thứ tự gốc, đáp án là `max(g)` (không phải `g[-1]`).
2. **Bước nhảy tối đa `d`** thay vì 1: `b = min(x, b + d)` (Bài tập 5).
3. **Được tăng cũng được giảm, chi phí = tổng thay đổi**: bài toán tối ưu chi phí khác hẳn (DP / tham lam với heap).
4. **945 Minimum Increment to Make Array Unique**, **1647 Minimum Deletions to Make Character Frequencies Unique**: sắp + tham lam "bậc thang".
5. **Trả về mảng kết quả** (Bài tập 4).

## 16. So sánh các cách làm

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Mọi hoán vị | Tham lam trên mỗi thứ tự | `O(n!·n)` | `O(n)` | Kiểm chứng |
| 2. Sắp + tham lam | Cầu thang `min(a[i], b+1)` | `O(n log n)` | `O(n)` | **Bản nộp** |
| 3. Đếm | Chặn bởi `n`, theo nhóm | `O(n)` | `O(n)` | Tối ưu tuyệt đối |
| 4. Nhị phân đáp án | `ok(M)`: `a[n−M+k−1] ≥ k` | `O(n log n)` | `O(n)` | Góc nhìn điều kiện cần/đủ |
| 5. Công thức đóng | `min(n, a[i] + n − 1 − i)` | `O(n log n)` | `O(n)` | Hiểu "phần tử nào chặn đỉnh" |

## 17. 5 bài tập vận dụng (từ dễ đến khó)

**Bài 1 (đoán output):** (a) `[1,1,1,1]`; (b) `[4,4,4,4]`; (c) `[2,1,4,1,5]`; (d) `[73,98,9]`.

**Bài 2 (tính tay):** Trace Cách 3 (đếm) cho `[6, 6, 1, 3, 2, 2]`.

**Bài 3 (điền TODO):** Hoàn thiện `staircase(a_sorted)` trả **toàn bộ** dãy `b` tham lam.

**Bài 4 (tự viết lại):** Viết `final_array(arr)` trả một mảng hợp lệ (sau sắp lại + giảm) đạt giá trị lớn nhất, kiểm tra bằng `valid` và mỗi phần tử không vượt phần tử gốc tương ứng (theo một ghép cặp).

**Bài 5 (mini challenge):** Viết `max_elem_step(arr, d)` khi điều kiện là `|b[i] − b[i−1]| ≤ d` (phần tử đầu vẫn là 1), đối chiếu vét cạn hoán vị.

In [24]:
# Bài 3 — TODO (chỉ định nghĩa, không gọi)
def staircase_todo(a_sorted):
    b = [1]
    for x in a_sorted[1:]:
        # TODO: thêm min(x, b[-1] + 1)
        pass
    return b

In [25]:
# Bài 4, 5 — khung tự viết (chỉ định nghĩa, không gọi)
def final_array(arr):
    # TODO: sắp tăng rồi trả staircase
    pass

def max_elem_step(arr, d):
    # TODO: b = 1; b = min(x, b + d)
    pass

## 18. Gợi ý 3 mức cho bài tập

**Bài 1.** Mức 1: sắp. Mức 2: cầu thang. Mức 3: (a) `1`; (b) `4`; (c) sắp `[1,1,2,4,5]` ⇒ `1,1,2,3,4` ⇒ `4`; (d) sắp `[9,73,98]` ⇒ `1,2,3` ⇒ `3`.

**Bài 2.** Mức 1: `n = 6`, `cnt`: 1×1, 2×2, 3×1, 6×2. Mức 2: `v=1: ans=1`; `v=2: min(2, 1+2) = 2`; `v=3: min(3, 3) = 3`. Mức 3: `v=6: min(6, 3+2) = 5` ⇒ đáp án `5`.

**Bài 3.** Mức 1: `b = [1]`. Mức 2: `b.append(min(x, b[-1] + 1))`. Mức 3: `b[-1]` là đáp án.

**Bài 4.** Mức 1: `a = sorted(arr)`. Mức 2: `b = staircase(a)`. Mức 3: ghép `b[i]` với `a[i]`: `b[i] ≤ a[i]`, `valid(b)`.

**Bài 5.** Mức 1: thay `+1` bằng `+d`. Mức 2: sắp tăng vẫn tối ưu (cùng lập luận). Mức 3: vét cạn hoán vị với luật `min(x, b + d)`.

## 19. Đáp án tham khảo cho bài tập

In [26]:
# Bài 1, 2, 3
f = Solution().maximumElementAfterDecrementingAndRearranging
b1 = [f(a) for a in [[1, 1, 1, 1], [4, 4, 4, 4], [2, 1, 4, 1, 5], [73, 98, 9]]]
print("Bài 1:", b1); assert b1 == [1, 4, 4, 3]
print("Bài 2:"); assert trace_count([6, 6, 1, 3, 2, 2]) == 5
def staircase_todo(a_sorted):
    b = [1]
    for x in a_sorted[1:]:
        b.append(min(x, b[-1] + 1))
    return b
assert staircase_todo([1, 2, 2, 3, 7, 9]) == [1, 2, 2, 3, 4, 5]
print("Bài 3: OK")

Bài 1: [1, 4, 4, 3]
Bài 2:
cnt (đã chặn bởi n): {1: 1, 2: 2, 3: 1, 6: 2}
  v=1: ans = min(1, 0 + 1) = 1
  v=2: ans = min(2, 1 + 2) = 2
  v=3: ans = min(3, 2 + 1) = 3
  v=6: ans = min(6, 3 + 2) = 5
Bài 3: OK


In [27]:
# Bài 4, 5
def final_array(arr):
    a = sorted(arr); return a, staircase_todo(a)

def max_elem_step(arr, d):
    a = sorted(arr); b = 1
    for x in a[1:]:
        b = min(x, b + d)
    return b

def step_brute(arr, d):
    best = 0
    for p in set(permutations(arr)):
        b = 1; mx = 1
        for x in p[1:]:
            b = min(x, b + d); mx = max(mx, b)
        best = max(best, mx)
    return best

rng = random.Random(4)
for _ in range(300):
    arr = [rng.randint(1, 12) for _ in range(rng.randint(1, 6))]
    a, b = final_array(arr)
    assert valid(b) and all(x <= y for x, y in zip(b, a)) and max(b) == f(arr)
    d = rng.randint(1, 4)
    assert max_elem_step(arr, d) == step_brute(arr, d)
print("Bài 4, 5: OK |", final_array([3, 1, 7, 2, 2, 9]), max_elem_step([100, 1, 1000], 3))

Bài 4, 5: OK | ([1, 2, 2, 3, 7, 9], [1, 2, 2, 3, 4, 5]) 7


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

**Template:**

```python
a = sorted(arr)
b = 1
for x in a[1:]:
    b = min(x, b + 1)
return b
# O(n): cnt[min(v, n)] += 1; ans = min(v, ans + cnt[v]) cho v = 1..n
```

**Checklist:**
- [ ] Đã **sắp tăng** trước?
- [ ] Phần tử đầu ép bằng **1**?
- [ ] `min(x, b + 1)` (không được tăng)?
- [ ] Bản đếm: chặn giá trị bởi `n`?
- [ ] Đã test `n = 1`, toàn số lớn, nhiều số 1, đã hợp lệ sẵn?